In [9]:
import pandas as pd
import numpy as np
from google.colab import drive

In [10]:
# accessing sample data from google drive
drive.mount('/content/drive')

# loading listings, calendar, reviews, and neighbourhood data
# los angeles, portland, salem, san diego

# change path depending on location in drive
path = "/content/drive/MyDrive/cs498_data"

data_tables = ["listings", "calendar", "reviews", "neighbourhoods"]
cities = ["los_angeles", "portland", "salem", "san_diego"]
datasets = {}

for city in cities:
    datasets[city] = {}
    for table in data_tables:
        if table == "neighbourhoods":
            fp = f"{path}/{city}/{table}.csv"
        else:
            fp = f"{path}/{city}/{table}.csv.gz"

        datasets[city][table] = pd.read_csv(fp)
        print(f"Read {fp} to dataframe.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Read /content/drive/MyDrive/cs498_data/los_angeles/listings.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/los_angeles/calendar.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/los_angeles/reviews.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/los_angeles/neighbourhoods.csv to dataframe.
Read /content/drive/MyDrive/cs498_data/portland/listings.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/portland/calendar.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/portland/reviews.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/portland/neighbourhoods.csv to dataframe.
Read /content/drive/MyDrive/cs498_data/salem/listings.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/salem/calendar.csv.gz to dataframe.
Read /content/drive/MyDrive/cs498_data/salem/reviews.csv.gz to dataframe.
Read /c

In [11]:
def neighbourhood_cleaning(datasets, city):
    datasets[city]['neighbourhoods']['city'] = city


In [12]:
def listings_cleaning(datasets, city):
    def clean_price(price):
      if pd.isna(price):
          return None
      else:
          return float(price.replace('$', '').replace(',',''))

    listings_columns = ['id','listing_url','name','description','host_id','host_name','neighbourhood_cleansed','property_type','room_type','accommodates','amenities','price','minimum_nights','maximum_nights','review_scores_rating']
    datasets[city]['listings'] = datasets[city]['listings'][listings_columns]
    listing_col_renamed = {
        'id': 'listing_id',
        'review_scores_raiting': 'rating',
        'neighbourhood_cleansed': 'neighbourhood'
}

    datasets[city]['listings'] = datasets[city]['listings'].rename(columns = listing_col_renamed)
    datasets[city]['listings']['price'] = datasets[city]['listings']['price'].apply(clean_price)
    datasets[city]['listings']['city'] = city
    datasets[city]['listings']['description'] = datasets[city]['listings']['description'].where(datasets[city]['listings']['description'].notna(), None)


In [13]:
def reviews_cleaning(datasets, city):
    datasets[city]['reviews']['city'] = city
    datasets[city]['reviews']['date'] = pd.to_datetime(datasets[city]['reviews']['date'], errors='coerce')
    datasets[city]['reviews']['month'] = datasets[city]['reviews']['date'].dt.month

In [14]:
def calendar_cleaning(datasets, city):
    datasets[city]['calendar']['city'] = city
    datasets[city]['calendar']['date'] = pd.to_datetime(datasets[city]['calendar']['date'], errors='coerce')
    datasets[city]['calendar']['month'] = datasets[city]['calendar']['date'].dt.month
    datasets[city]['calendar']['available'] = datasets[city]['calendar']['available'].map({'t': True,'f': False})

In [15]:
for city in cities:
    calendar_cleaning(datasets, city)
    listings_cleaning(datasets, city)
    reviews_cleaning(datasets, city)
    neighbourhood_cleaning(datasets, city)

In [19]:
# example listings
display(datasets['los_angeles']['listings'].head())
datasets['los_angeles']['listings'].info()

,listing_id,listing_url,name,description,host_id,host_name,neighbourhood,property_type,room_type,accommodates,amenities,price,minimum_nights,maximum_nights,review_scores_rating,city
0,2708,https://www.airbnb.com/rooms/2708,"Run Runyon, Beaut Furn Mirror Mini-Suit w/ Fir...",Run Runyon Canyon<br /><br />Gym & Sauna <br /...,3008,⁨Chas.⁩,Hollywood,Private room in rental unit,Private room,1,"[""1801 Beekman Luxury Spa Goat Milk Soap & Han...",76.31,30.0,1125.0,4.87,los_angeles
1,2732,https://www.airbnb.com/rooms/2732,Zen Life at the Beach,An oasis of tranquility awaits you.,3041,Yogapriestess,Santa Monica,Private room in rental unit,Private room,1,"[""Refrigerator"", ""Private patio or balcony"", ""...",215.21,7.0,27.0,4.41,los_angeles
2,6033,https://www.airbnb.com/rooms/6033,Poolside Serenity Studio,Our distinctive bachelor's studio for 1-3 gues...,11619,Sarah,Woodland Hills,Entire bungalow,Entire home/apt,3,"[""Children\u2019s books and toys"", ""Hot water""...",99.40,30.0,1125.0,4.38,los_angeles
3,6931,https://www.airbnb.com/rooms/6931,"RUN Runyon, Beau Furn Rms Terrace Hollyw Hill ...",Run Runyon Canyon & Views<br /><br />Gym & Sau...,3008,⁨Chas.⁩,Hollywood,Private room in rental unit,Private room,1,"[""Baking sheet"", ""Refrigerator"", ""Private pati...",69.50,30.0,1125.0,4.87,los_angeles
4,152514,https://www.airbnb.com/rooms/152514,Charming Guest House | 1 block to beach!,Wake up to the sound of waves crashing on the ...,733870,Jeanette,Manhattan Beach,Entire guesthouse,Entire home/apt,2,"[""Baking sheet"", ""Refrigerator"", ""Hot water"", ...",173.25,30.0,1125.0,5.00,los_angeles


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43781 entries, 0 to 43780
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   listing_id            43781 non-null  int64  
 1   listing_url           43781 non-null  object 
 2   name                  43781 non-null  object 
 3   description           43541 non-null  object 
 4   host_id               43781 non-null  int64  
 5   host_name             43781 non-null  object 
 6   neighbourhood         43781 non-null  object 
 7   property_type         43781 non-null  object 
 8   room_type             43781 non-null  object 
 9   accommodates          43781 non-null  int64  
 10  amenities             43781 non-null  object 
 11  price                 36687 non-null  float64
 12  minimum_nights        43761 non-null  float64
 13  maximum_nights        43761 non-null  float64
 14  review_scores_rating  32936 non-null  float64
 15  city               

In [20]:
# example reviews
display(datasets['los_angeles']['reviews'].head())
datasets['los_angeles']['reviews'].info()

,listing_id,id,date,reviewer_id,reviewer_name,comments,city,month
0,2708,13994902,2014-06-09,10905424,Kuberan,i had a wonderful stay. Everything from start ...,los_angeles,6
1,2708,14606598,2014-06-23,2247288,Camilla,Charles is just amazing and he made my stay sp...,los_angeles,6
2,2708,39597339,2015-07-25,27974696,Fallon,Staying with Chas was an absolute pleasure. He...,los_angeles,7
3,2708,61157407,2016-02-01,33226412,Haroon,Charles is a most wonderful host. I enjoyed my...,los_angeles,2
4,2708,66196280,2016-03-20,23408691,Massimo Litterio,Chas is a really good host. He gives me a lot ...,los_angeles,3


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1890200 entries, 0 to 1890199
Data columns (total 8 columns):
 #   Column         Dtype         
---  ------         -----         
 0   listing_id     int64         
 1   id             int64         
 2   date           datetime64[ns]
 3   reviewer_id    int64         
 4   reviewer_name  object        
 5   comments       object        
 6   city           object        
 7   month          int32         
dtypes: datetime64[ns](1), int32(1), int64(3), object(3)
memory usage: 108.2+ MB


In [17]:
# example calendar
display(datasets['los_angeles']['calendar'].head())
datasets['los_angeles']['calendar'].info()

,listing_id,date,available,minimum_nights,maximum_nights,city,month
0,2708,2026-09-15,False,30,1125,los_angeles,9
1,2708,2026-09-16,False,30,1125,los_angeles,9
2,2708,2026-09-17,False,30,1125,los_angeles,9
3,2708,2026-09-18,False,30,1125,los_angeles,9
4,2708,2026-09-19,False,30,1125,los_angeles,9


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15980081 entries, 0 to 15980080
Data columns (total 7 columns):
 #   Column          Dtype         
---  ------          -----         
 0   listing_id      int64         
 1   date            datetime64[ns]
 2   available       bool          
 3   minimum_nights  int64         
 4   maximum_nights  int64         
 5   city            object        
 6   month           int32         
dtypes: bool(1), datetime64[ns](1), int32(1), int64(3), object(1)
memory usage: 685.8+ MB


In [21]:
# example neighborhoods
display(datasets['los_angeles']['neighbourhoods'].head())
datasets['los_angeles']['neighbourhoods'].info()

,neighbourhood_group,neighbourhood,city
0,City of Los Angeles,Adams-Normandie,los_angeles
1,City of Los Angeles,Arleta,los_angeles
2,City of Los Angeles,Arlington Heights,los_angeles
3,City of Los Angeles,Atwater Village,los_angeles
4,City of Los Angeles,Baldwin Hills/Crenshaw,los_angeles


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   neighbourhood_group  270 non-null    object
 1   neighbourhood        270 non-null    object
 2   city                 270 non-null    object
dtypes: object(3)
memory usage: 6.5+ KB


In [22]:
final_datasets = {}

for table in data_tables:
    final_datasets[table] = pd.concat([datasets[city][table] for city in cities], ignore_index=True)

In [23]:
for table, df in final_datasets.items():
    size_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    print(f"{table}: {size_mb:.2f} MB")

listings: 114.44 MB
calendar: 2063.54 MB
reviews: 1711.00 MB
neighbourhoods: 0.08 MB


In [29]:
# testing to see if data can fit into mongodb atlas
import os
import gzip

output_dir = "/content/drive/MyDrive/airbnb_cleaned"
os.makedirs(output_dir, exist_ok=True)

for table, df in final_datasets.items():

    filepath = f"{output_dir}/{table}.jsonl.gz"

    with gzip.open(filepath, "wt", encoding="utf-8") as f:
        for start in range(0, len(df), 100000):
            chunk = df.iloc[start:start + 100000]

            f.write(chunk.to_json(
                orient="records",
                lines=True,
                date_format="iso"
            ))

    print(f"Exported {table}: {len(df):,} records")

Exported listings: 61,538 records
Exported calendar: 22,461,452 records
Exported reviews: 3,408,752 records
Exported neighbourhoods: 481 records


In [30]:
# testing if json especially other cities appear
import gzip
import json

filepath = "/content/drive/MyDrive/airbnb_cleaned/listings.jsonl.gz"

with gzip.open(filepath, "rt", encoding="utf-8") as f:
    for line in f:
        document = json.loads(line)

        if document['city'] == 'portland':
            print(json.dumps(document, indent=2))
            break

{
  "listing_id": 35418187,
  "listing_url": "https://www.airbnb.com/rooms/35418187",
  "name": "Forest Studio in Portland w/ yurt",
  "description": "Rustic retreat 10\u201315 min from downtown Portland. Includes a yurt with electricity, Wi-Fi, and simple comforts, plus a private basement studio with its own entrance, kitchen, bath w/ warm shower. Not luxury glamping\u2014no polished floors or paved trails\u2014just a peaceful spot near gardens, trails, and St. Johns. <br /><br />Note: Studio is 50 yards from yurt across grass and pavement; not ideal for guests with mobility challenges.  Also the space is near forest park so you should be okay driving on a hill.",
  "host_id": 3005870,
  "host_name": "Jeremiah",
  "neighbourhood": "Linnton",
  "property_type": "Yurt",
  "room_type": "Entire home/apt",
  "accommodates": 4,
  "amenities": "[\"Coffee maker\", \"Extra pillows and blankets\", \"Shared backyard \\u2013 Not fully fenced\", \"Self check-in\", \"Luggage dropoff allowed\", \"Ca